In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("../Dataset/master_fataka_sales.csv")
df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year

annual_product_sales = df.groupby(["Product_ID", "Product_Name", "Category", "Unit_Price", "Year"]).agg(
    Units_Sold=("Quantity_Sold", "sum")
).reset_index()

pivot_sales = annual_product_sales.pivot_table(
    index=["Product_ID", "Product_Name", "Category", "Unit_Price"],
    columns="Year",
    values="Units_Sold",
    fill_value=0
).reset_index()

pivot_sales.head()

Year,Product_ID,Product_Name,Category,Unit_Price,2023,2024,2025,2026,2027
0,P101,Sparklers 10cm (Box),Sparklers,40,707.0,617.0,632.0,724.0,721.0
1,P102,Sparklers 30cm (Box),Sparklers,90,719.0,752.0,689.0,738.0,656.0
2,P103,Electric Sparklers Multicolor,Sparklers,120,607.0,658.0,728.0,701.0,640.0
3,P104,Peacock Feather Sparklers,Sparklers,150,746.0,807.0,761.0,671.0,563.0
4,P201,Flower Pots Small,Ground Novelties,180,663.0,714.0,747.0,632.0,799.0



Builds a predictive demand forecasting model and generates a stock optimization matrix for the 21 firecracker products based on 5-year historical trends.


In [5]:
years = sorted(df["Year"].unique())
latest_year = years[-1]
prev_year = years[-2] if len(years) > 1 else latest_year

recommendations = []
for _, row in pivot_sales.iterrows():
    recent_sales = row.get(latest_year, 0)
    older_sales = row.get(prev_year, recent_sales)

    yoy_growth = (recent_sales - older_sales) / older_sales if older_sales > 0 else 0.0
    projected_growth_factor = 1.0 + (yoy_growth * 0.5)
    predicted_demand = int(max(recent_sales * projected_growth_factor, 5))

    if yoy_growth > 0.10:
        action = "Increase Stock (+15%)"
    elif yoy_growth < -0.10:
        action = "Reduce Stock (-15%)"
    else:
        action = "Maintain Stock (Stable)"

    recommended_stock = int(predicted_demand * 1.10)
    wholesale_cost = recommended_stock * row["Unit_Price"] * 0.6

    recommendations.append({
        "Product_Name": row["Product_Name"],
        "Category": row["Category"],
        f"Sales_{latest_year}": recent_sales,
        "Predicted_Demand_Next_Year": predicted_demand,
        "Recommended_Stock_Order": recommended_stock,
        "Stock_Action": action,
        "Estimated_Wholesale_Investment_₹": round(wholesale_cost, 2)
    })

rec_df = pd.DataFrame(recommendations).sort_values(by="Recommended_Stock_Order", ascending=False).reset_index(drop=True)

# Export action plan
rec_df.to_csv("../Dataset/inventory_action_plan.csv", index=False)
display(rec_df.head(10))

,Product_Name,Category,Sales_2027,Predicted_Demand_Next_Year,Recommended_Stock_Order,Stock_Action,Estimated_Wholesale_Investment_₹
0,Flower Pots Small,Ground Novelties,799.0,904,994,Increase Stock (+15%),107352.0
1,Atom Bomb Deluxe,Bombs,750.0,765,841,Maintain Stock (Stable),111012.0
2,Magic Pencil,Novelties,722.0,751,826,Maintain Stock (Stable),79296.0
3,Fancy Sky Shot 30-Shot,Aerial Shots,653.0,728,800,Increase Stock (+15%),864000.0
4,Two Sound Rocket,Rockets,694.0,728,800,Maintain Stock (Stable),124800.0
5,Sparkler Cones,Novelties,696.0,726,798,Maintain Stock (Stable),143640.0
6,Sparklers 10cm (Box),Sparklers,721.0,719,790,Maintain Stock (Stable),18960.0
7,Rocket Bomb Single,Rockets,702.0,713,784,Maintain Stock (Stable),61152.0
8,Atomic Rocket,Rockets,723.0,708,778,Maintain Stock (Stable),186720.0
9,Fancy Sky Shot 12-Shot,Aerial Shots,677.0,707,777,Maintain Stock (Stable),349650.0
